# 03. Baseline: Dummy Classifier và Logistic Regression

**Mục tiêu (T09):** ghép các thành phần T06–T08 và chạy quy trình baseline từ đầu đến cuối.

Quy trình: `load_raw()` → `clean_data()` → `add_features()` → `temporal_split()` → Pipeline tiền xử lý + mô hình → CV theo thời gian trên train → đánh giá validation → ghi `reports/results/results.csv`.

Notebook chỉ dùng train để cross-validation và validation để đánh giá baseline. **Không đánh giá hoặc lựa chọn mô hình trên test** ở bước này.


## 0. Chuẩn bị môi trường và đường dẫn

In [1]:
import sys
from pathlib import Path

# Notebook có thể được chạy từ thư mục gốc hoặc từ notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Tất cả đường dẫn đầu ra đều tính từ thư mục gốc dự án.
PROJECT_ROOT = ROOT
assert (PROJECT_ROOT / "src").is_dir(), (
    f"Không tìm thấy thư mục src. CWD={Path.cwd()} | ROOT={PROJECT_ROOT}. "
    "Hãy mở đúng thư mục dự án HotelBookingML trong VS Code."
)

RESULTS_PATH = PROJECT_ROOT / "reports" / "results" / "results.csv"
print("Working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Results path:", RESULTS_PATH)


Working directory: c:\Users\Hi\OneDrive\Desktop\ML\btl\HotelBookingML\notebooks
Project root: C:\Users\Hi\OneDrive\Desktop\ML\btl\HotelBookingML
Results path: C:\Users\Hi\OneDrive\Desktop\ML\btl\HotelBookingML\reports\results\results.csv


In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from src.config import RANDOM_STATE, TARGET_COLUMN, CLEAN_CATEGORICAL_FEATURES
from src.data import load_raw
from src.preprocess import clean_data, build_model_pipeline
from src.features import add_features, ENGINEERED_CATEGORICAL_FEATURES, ENGINEERED_NUMERIC_FEATURES
from src.split import temporal_split
from src.evaluate import (
    RESULT_COLUMNS, evaluate_classification, time_series_cv, append_result
)

TARGET = TARGET_COLUMN
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)


## 1. Đọc và chuẩn bị dữ liệu (T07–T08)

In [3]:
raw = load_raw()
clean = clean_data(raw)
df = add_features(clean)

print(f"Raw data:   {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Clean data: {clean.shape[0]:,} rows x {clean.shape[1]} columns")
print(f"Featured:   {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Rows removed by clean_data: {len(raw) - len(clean):,}")
print(f"Cancellation rate: {df[TARGET].mean():.2%}")


Raw data:   119,390 rows x 32 columns
Clean data: 119,205 rows x 33 columns
Featured:   119,205 rows x 34 columns
Rows removed by clean_data: 185
Cancellation rate: 37.08%


## 2. Khai báo bộ đặc trưng và chia dữ liệu theo thời gian

In [4]:
# Bộ clean 22 cột đã chốt ở M1; arrival_weekday là đặc trưng T08 thêm vào.
CLEAN_NUMERIC_FEATURES = [
    "lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "total_of_special_requests",
]
NUMERIC_FEATURES = CLEAN_NUMERIC_FEATURES + list(ENGINEERED_NUMERIC_FEATURES)
CATEGORICAL_FEATURES = list(CLEAN_CATEGORICAL_FEATURES) + list(ENGINEERED_CATEGORICAL_FEATURES)
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

assert len(FEATURES) == 23, f"Expected 23 features, got {len(FEATURES)}"
assert len(set(FEATURES)) == len(FEATURES), "Feature list contains duplicates"
FORBIDDEN = {TARGET, "reservation_status", "reservation_status_date", "arrival_date"}
assert not (set(FEATURES) & FORBIDDEN), "Leakage/target column found in features"
missing_features = set(FEATURES) - set(df.columns)
assert not missing_features, f"Missing features: {sorted(missing_features)}"

train, valid, test = temporal_split(
    df, train_end="2017-01-01", valid_end="2017-07-01", date_col="arrival_date"
)

split_summary = pd.DataFrame([
    {"split": name, "rows": len(part), "cancel_rate": part[TARGET].mean(),
     "arrival_from": part["arrival_date"].min().date(),
     "arrival_to": part["arrival_date"].max().date()}
    for name, part in [("train", train), ("validation", valid), ("test (held out)", test)]
]).set_index("split")
display(split_summary)
print(f"Features: {len(FEATURES)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")


,rows,cancel_rate,arrival_from,arrival_to
split,,,,
train,78586,0.3622,2015-07-01,2016-12-31
validation,30386,0.3928,2017-01-01,2017-06-30
test (held out),10233,0.3712,2017-07-01,2017-08-31


Features: 23 (13 numeric, 10 categorical)


## 3. Tạo X/y và kiểm tra pipeline

In [5]:
X_train = train[FEATURES]
y_train = train[TARGET]
X_valid = valid[FEATURES]
y_valid = valid[TARGET]

print("Train:", X_train.shape, "Validation:", X_valid.shape)
print("Categorical features:", CATEGORICAL_FEATURES)


Train: (78586, 23) Validation: (30386, 23)
Categorical features: ['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'customer_type', 'agent', 'company', 'arrival_weekday']


## 4. Baseline models: Dummy và Logistic Regression

In [6]:
models = {
    "DummyClassifier": (
        DummyClassifier(strategy="prior"),
        False,
    ),
    "LogisticRegression": (
        LogisticRegression(
            max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE
        ),
        True,
    ),
}

pipelines = {}
for model_name, (estimator, scale_numeric) in models.items():
    pipelines[model_name] = build_model_pipeline(
        X_train, estimator,
        numeric_features=NUMERIC_FEATURES,
        categorical_features=CATEGORICAL_FEATURES,
        scale_numeric=scale_numeric,
    )

print("Đã tạo pipeline cho:", list(pipelines))


Đã tạo pipeline cho: ['DummyClassifier', 'LogisticRegression']


## 5. Cross-validation theo thời gian trên train

In [7]:
cv_summaries = []
cv_details = {}
for model_name, pipeline in pipelines.items():
    scores = time_series_cv(
        pipeline, X_train, y_train, n_splits=5, scoring="average_precision"
    )
    cv_details[model_name] = scores
    cv_summaries.append({
        "model_name": model_name,
        "average_precision_mean": scores["score"].mean(),
        "average_precision_std": scores["score"].std(ddof=1),
    })
    print(f"\n{model_name}")
    display(scores)

cv_summary = pd.DataFrame(cv_summaries).set_index("model_name")
display(cv_summary)



DummyClassifier


,fold,train_size,validation_size,score,scoring
0,1,13101,13097,0.2882,average_precision
1,2,26198,13097,0.3547,average_precision
2,3,39295,13097,0.3674,average_precision
3,4,52392,13097,0.3668,average_precision
4,5,65489,13097,0.3785,average_precision



LogisticRegression


,fold,train_size,validation_size,score,scoring
0,1,13101,13097,0.7397,average_precision
1,2,26198,13097,0.7703,average_precision
2,3,39295,13097,0.7432,average_precision
3,4,52392,13097,0.7511,average_precision
4,5,65489,13097,0.7576,average_precision


,average_precision_mean,average_precision_std
model_name,,
DummyClassifier,0.3511,0.0362
LogisticRegression,0.7524,0.0122


## 6. Fit trên toàn bộ train và đánh giá validation

In [8]:
validation_results = []
for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_valid)
    y_score = pipeline.predict_proba(X_valid)[:, 1]
    metrics = evaluate_classification(y_valid, y_pred, y_score)
    row = {
        "model_name": model_name,
        "n_eval": len(valid),
        "eval_cancel_rate": float(y_valid.mean()),
        **metrics,
    }
    validation_results.append(row)

validation_summary = pd.DataFrame(validation_results).set_index("model_name")
display(validation_summary)


,n_eval,eval_cancel_rate,average_precision,f1,roc_auc,precision,recall,TN,FP,FN,TP
model_name,,,,,,,,,,,
DummyClassifier,30386,0.3928,0.3928,0.0000,0.5000,0.0000,0.0000,18450.0000,0.0000,11936.0000,0.0000
LogisticRegression,30386,0.3928,0.7862,0.6586,0.8224,0.7593,0.5814,16250.0000,2200.0000,4996.0000,6940.0000


## 7. Ghi kết quả theo schema chung của T06

In [9]:
# Khi chạy lại notebook, thay thế các dòng baseline T09 cũ của đúng thí nghiệm này
# để tránh append trùng kết quả; các kết quả thí nghiệm khác được giữ nguyên.
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
if RESULTS_PATH.exists() and RESULTS_PATH.stat().st_size > 0:
    existing = pd.read_csv(RESULTS_PATH)
    if list(existing.columns) != RESULT_COLUMNS:
        raise ValueError("results.csv không khớp RESULT_COLUMNS trong src/evaluate.py")
    is_this_baseline = (
        existing["model_name"].isin(list(models))
        & existing["experiment"].eq("main")
        & existing["feature_set"].eq("clean")
        & existing["split"].eq("temporal")
        & existing["train_hotel"].eq("all")
        & existing["eval_hotel"].eq("all")
        & existing["eval_set"].isin(["cv", "validation"])
    )
    existing.loc[~is_this_baseline].to_csv(RESULTS_PATH, index=False)

for model_name, scores in cv_details.items():
    append_result({
        "model_name": model_name,
        "experiment": "main",
        "feature_set": "clean",
        "split": "temporal",
        "train_hotel": "all",
        "eval_hotel": "all",
        "eval_set": "cv",
        "average_precision": float(scores["score"].mean()),
        "average_precision_std": float(scores["score"].std(ddof=1)),
        "params": json.dumps(pipelines[model_name].named_steps["model"].get_params(), default=str, sort_keys=True),
    }, path=str(RESULTS_PATH))

for row in validation_results:
    model_name = row["model_name"]
    append_result({
        "model_name": model_name,
        "experiment": "main",
        "feature_set": "clean",
        "split": "temporal",
        "train_hotel": "all",
        "eval_hotel": "all",
        "eval_set": "validation",
        "n_eval": row["n_eval"],
        "eval_cancel_rate": row["eval_cancel_rate"],
        "average_precision": row["average_precision"],
        "f1": row["f1"],
        "roc_auc": row["roc_auc"],
        "precision": row["precision"],
        "recall": row["recall"],
        "TN": row["TN"], "FP": row["FP"], "FN": row["FN"], "TP": row["TP"],
        "threshold": 0.5,
        "params": json.dumps(pipelines[model_name].named_steps["model"].get_params(), default=str, sort_keys=True),
    }, path=str(RESULTS_PATH))

results = pd.read_csv(RESULTS_PATH)
print("Saved:", RESULTS_PATH)
display(results.tail(4))


Saved: C:\Users\Hi\OneDrive\Desktop\ML\btl\HotelBookingML\reports\results\results.csv


,model_name,experiment,feature_set,split,train_hotel,eval_hotel,eval_set,n_eval,eval_cancel_rate,average_precision,average_precision_std,f1,roc_auc,precision,recall,threshold,cost_scenario,C_FN,C_FP,TN,FP,FN,TP,expected_cost,cost_per_booking,params
0,DummyClassifier,main,clean,temporal,all,all,cv,NaN,NaN,0.3511,0.0362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""constant"": null, ""random_state"": null, ""stra..."
1,LogisticRegression,main,clean,temporal,all,all,cv,NaN,NaN,0.7524,0.0122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{""C"": 1.0, ""class_weight"": null, ""dual"": false..."
2,DummyClassifier,main,clean,temporal,all,all,validation,30386.0000,0.3928,0.3928,NaN,0.0000,0.5000,0.0000,0.0000,0.5000,NaN,NaN,NaN,18450.0000,0.0000,11936.0000,0.0000,NaN,NaN,"{""constant"": null, ""random_state"": null, ""stra..."
3,LogisticRegression,main,clean,temporal,all,all,validation,30386.0000,0.3928,0.7862,NaN,0.6586,0.8224,0.7593,0.5814,0.5000,NaN,NaN,NaN,16250.0000,2200.0000,4996.0000,6940.0000,NaN,NaN,"{""C"": 1.0, ""class_weight"": null, ""dual"": false..."


## 8. Kết luận baseline và lưu ý

- DummyClassifier là mốc tham chiếu; với `strategy="prior"`, xác suất dự đoán là hằng số theo tỉ lệ lớp trong train nên F1 ở ngưỡng 0,5 có thể bằng 0.
- Logistic Regression là baseline học được quan hệ giữa các đặc trưng và xác suất hủy.
- PR-AUC (`average_precision`) là metric chính; F1, ROC-AUC, Precision và Recall là metric bổ sung.
- Tập test được giữ nguyên, chưa dùng trong T09. Chỉ dùng cho đánh giá cuối theo kế hoạch của nhóm.
- Pipeline chứa imputer, encoder và scaler; các bước này chỉ được fit trên phần train tương ứng (kể cả trong từng fold CV).
